# Работа с крупномасштабными базами знаний


**Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)**

Материалы:

- Курс лекций [Semantic Technologies for Developers](https://yadi.sk/d/SepS4JBLhDYy4Q), лекция №07
- https://query.wikidata.org/
- https://www.wikidata.org/wiki/Wikidata:SPARQL_tutorial
- https://dbpedia.org/sparql
- https://www.dbpedia.org/about/
- https://www.wikidata.org/wiki/Wikidata:SPARQL_federation_input
- https://www.w3.org/TR/sparql11-federated-query/
- https://www.wikidata.org/wiki/Wikidata:Data_access/ru
- https://www.wikidata.org/wiki/Special:EntityData/Q42.ttl
- https://www.wikidata.org/wiki/Wikidata:SPARQL_tutorial


## Задачи для самостоятельного решения


In [ ]:
!pip install sparqlwrapper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.0/569.0 kB 21.0 MB/s eta 0:00:00


<p class="task" id="1"></p>

1\. Используя информацию из Wikidata, найдите все картины, созданные в XV веке, на которых изображены здания. Для каждой картины укажите:

- название картины и художника;
- год создания картины;
- название и местоположение изображённого здания.

Для выполнения воспользуйтесь [точкой доступа Wikidata](https://query.wikidata.org/). Выполните запрос и прикрепите скриншот с результатом работы сервиса. Экспортируйте результаты выполнения запроса и представьте результаты в виде `pd.DataFrame`.

- [ ] Проверено на семинаре


In [2]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

In [9]:
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setQuery("""
SELECT ?painting ?paintingLabel ?artist ?artistLabel ?year ?building ?buildingLabel ?buildingLocationLabel WHERE {
  ?painting wdt:P31/wdt:P279* wd:Q3305213 .
  ?painting wdt:P571 ?date .
  BIND(YEAR(?date) AS ?year)
  FILTER(?year >= 1401 && ?year <= 1500)

  ?painting wdt:P180 ?building .
  ?building wdt:P31 wd:Q41176 .
  ?painting wdt:P170 ?artist .

  OPTIONAL { ?building wdt:P276 ?buildingLocation . }

  SERVICE wikibase:label { bd:serviceParam wikibase:language "ru,en". }
}
ORDER BY ?year
LIMIT 50
""")

In [11]:
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

records = []
for result in results["results"]["bindings"]:
    records.append(
        {
            "Painting": result["paintingLabel"]["value"],
            "Artist": result["artistLabel"]["value"],
            "Year": result["year"]["value"],
            "Building": result["buildingLabel"]["value"],
        }
    )

df = pd.DataFrame(records); df.head()

,Painting,Artist,Year,Building
0,May,Братья Лимбурги,1410,Большой Шатле
1,Confirmation of the Franciscan Rule,Доменико Гирландайо,1483,Лоджия Ланци


<p class="task" id="2"></p>

2\. Используя информацию из Wikidata, найдите города, которые удовлетворяют следующим условиям:

- население за последний известный период больше 100 000 человек;
- расположены на высоте более 2000 метров над уровнем моря.

Для каждого города укажите:

- Название и страну;
- Точную высоту над уровнем моря;
- Население (с годом последней переписи).

Для выполнения воспользуйтесь пакетом `SPARQLWrapper`. Выполните запрос и представьте результаты в виде `pd.DataFrame`.

- [ ] Проверено на семинаре


In [ ]:
query = """
'''Города: elevation > 2000 м, population (wdt) > 100000,
и обязательно год переписи (pq:P585) для той же записи о населении.'''
SELECT
  ?city ?cityLabel ?countryLabel ?elevation ?population ?populationYear
WHERE {
  # быстрые wdt-триплеты — используются индексы
  ?city wdt:P31 wd:Q515 ;        # city
        wdt:P2044 ?elevation ;   # elevation (meters)
        wdt:P17 ?country ;       # country
        wdt:P1082 ?population .  # population (preferred/simple value)

  # ранние фильтры для скорости
  FILTER(?elevation > 2000)
  FILTER(xsd:decimal(?population) > 100000)

  # ОБЯЗАТЕЛЬНО: найдём statement о населении с qualifier pq:P585 (point in time)
  # и значением, совпадающим с wdt:P1082
  ?city p:P1082 ?popStmt .
  ?popStmt ps:P1082 ?popValue .
  ?popStmt pq:P585 ?popDate .
  FILTER(xsd:decimal(?popValue) = xsd:decimal(?population))
  BIND(YEAR(?popDate) AS ?populationYear)

  SERVICE wikibase:label { bd:serviceParam wikibase:language "ru,en". }
}
ORDER BY DESC(xsd:decimal(?population))
LIMIT 200
"""

In [23]:
sparql.setQuery(query)
sparql.setReturnFormat(JSON)

results = sparql.query().convert()

records = []
for result in results["results"]["bindings"]:
    records.append(
        {
            "City": result["cityLabel"]["value"],
            "Country": result["countryLabel"]["value"],
            "Elevation (m)": float(result["elevation"]["value"]),
            "Population": int(result["population"]["value"]),
            "Year": int(result["populationYear"]["value"]),
        }
    )

df = pd.DataFrame(records); df.head()

,City,Country,Elevation (m),Population,Year
0,Мехико,Мексика,2240.0,9209944,2020
1,Сана,Йемен,2150.0,2957000,2015
2,Сана,Йемен,2253.0,2957000,2015
3,Синин,Китай,2275.0,2467965,2020
4,Синин,Китай,2275.0,2208708,2010


<p class="task" id="3"></p>

3\. Используя информацию из DBPedia, найдите актёров, которые снимались в фильмах одного режиссёра на протяжении не менее 20 лет. Для каждой пары актёр-режиссёр укажите:

- Имена актёра и режиссёра;
- Список всех совместных фильмов ;
- Дату первого совместного фильма;
- Дату последнего совместного фильма.

Для выполнения воспользуйтесь [точкой доступа DBPedia](https://dbpedia.org/sparql). Выполните запрос и прикрепите скриншот с результатом работы сервиса. Экспортируйте результаты выполнения запроса и представьте результаты в виде `pd.DataFrame`.

- [ ] Проверено на семинаре


In [46]:
query = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dbp: <http://dbpedia.org/property/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?film ?filmLabel ?actor ?actorLabel ?director ?directorLabel ?releaseDate WHERE {
  ?film a dbo:Film ;
        dbo:starring ?actor ;
        dbo:director ?director ;
        rdfs:label ?filmLabel .
  # учёт разных предикатов для даты релиза
  { ?film dbo:releaseDate ?releaseDate }
  UNION
  { ?film dbp:released ?releaseDate }
  # метки только на английском (убирает лишние дубляжи)
  FILTER(lang(?filmLabel) = "en")
  FILTER(lang(?actorLabel) = "en")
  FILTER(lang(?directorLabel) = "en")
  ?actor rdfs:label ?actorLabel .
  ?director rdfs:label ?directorLabel .
  # год должен быть читаемым — первые 4 символа цифры
  FILTER(regex(STR(?releaseDate), "^[0-9]{4}"))
}
ORDER BY ?actorLabel ?directorLabel ?releaseDate
"""

In [47]:
from collections import defaultdict

In [48]:
sparql = SPARQLWrapper("https://dbpedia.org/sparql")
sparql.setReturnFormat(JSON)
sparql.setQuery(query)
results = sparql.query().convert()

# Сбор данных в Python
pairs = defaultdict(list)
for r in results["results"]["bindings"]:
    actor = r["actorLabel"]["value"]
    director = r["directorLabel"]["value"]
    film = r["filmLabel"]["value"]
    year = r["releaseDate"]["value"][:4]  # год
    pairs[(actor, director)].append((film, year))

# Формирование итогового DataFrame
data = []
for (actor, director), films in pairs.items():
    years = [int(y) for f, y in films if y.isdigit()]
    if years and max(years) - min(years) >= 20:
        film_list = ", ".join(f for f, y in films)
        data.append([actor, director, film_list, min(years), max(years)])

df = pd.DataFrame(
    data,
    columns=["Actor", "Director", "Films", "FirstFilmYear", "LastFilmYear"]
)
df.head()

,Actor,Director,Films,FirstFilmYear,LastFilmYear
0,Akkineni Nageswara Rao,K. Viswanath,"Aatma Gowravam, Sutradharulu",1966,1989
1,Alberto Sordi,Luigi Magni,"The Conspirators (1969 film), The Conspirators...",1969,1998
2,Amanda Bearse,Tom Holland (filmmaker),"Fright Night, Fright Night",1985,2011
3,Amanda Root,Brian Cosgrove,"The BFG (1989 film), The BFG (1989 film)",1989,2016
4,Ambareesh,Rajendra Singh Babu,"Bhaari Bharjari Bete, Thipparalli Tharlegalu",1981,2010


<p class="task" id="4"></p>

4\. Используя информацию из DBPedia, найдите топ-5 городов в Европе (исключая столицы) с наибольшим население. Для каждого города укажите его название, страну, население и дату основания.

Для выполнения воспользуйтесь пакетом `SPARQLWrapper`. Выполните запрос и представьте результаты в виде `pd.DataFrame`.

- [ ] Проверено на семинаре


In [50]:
sparql = SPARQLWrapper("https://dbpedia.org/sparql")
sparql.setReturnFormat(JSON)

In [51]:
query = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?city ?cityLabel ?countryLabel ?population ?foundingDate
WHERE {
  ?city a dbo:City ;
        dbo:country ?country ;
        dbo:populationTotal ?population ;
        dbo:foundingDate ?foundingDate .
  ?city rdfs:label ?cityLabel .
  ?country rdfs:label ?countryLabel .

  FILTER(lang(?cityLabel) = "en" && lang(?countryLabel) = "en")

  # Географические границы Европы (широта и долгота)
  ?city geo:lat ?lat .
  ?city geo:long ?long .
  FILTER(?lat > 35 && ?lat < 71 && ?long > -10 && ?long < 40)

  # Исключаем столицы
  FILTER NOT EXISTS { ?country dbo:capital ?city }
}
ORDER BY DESC(?population)
LIMIT 5
"""

In [52]:
sparql.setQuery(query)
results = sparql.query().convert()

data = []
for r in results["results"]["bindings"]:
    city = r["cityLabel"]["value"]
    country = r["countryLabel"]["value"]
    population = r["population"]["value"]
    founding = r.get("foundingDate", {}).get("value", "")
    data.append([city, country, population, founding])

df = pd.DataFrame(data, columns=["Город", "Страна", "Население", "Дата основания"])
df.head()

,Город,Страна,Население,Дата основания
0,Saint Petersburg,Russia,5601911,1703-05-27
1,Metropolitan City of Catania,Italy,1068563,2015-08-04
2,Metropolitan City of Florence,Italy,989460,2015-01-01
3,Metropolitan City of Venice,Italy,833934,2015-01-01
4,Metropolitan City of Genoa,Italy,818651,2015-01-01


<p class="task" id="5"></p>

5\. Напишите объединенный запрос для получения информации из Wikidata и DPBedia одновременно.

Запрос должен состоять из двух (условных) частей:

1. При помощи директивы SERVICE обращаетесь к DBPedia и получаете набор из 100 городов России с их названием на английском языке, численностью населения и ссылкой на аналогичную сущность в Wikidata
2. Используя ссылку на сущность из Wikidata получаете численность населения за последний актуальный период .

В результате запрос должен возвращать информацию о численности из двух источников.

Для выполнения воспользуйтесь [точкой доступа Wikidata](https://query.wikidata.org/). Выполните запрос и прикрепите скриншот с результатом работы сервиса. Экспортируйте результаты выполнения запроса и представьте результаты в виде `pd.DataFrame`.

При возникновении проблем с длительностью выполнения запросов вы можете ограничить количество строк в результате или сделать некоторые поля опциональными.

- [ ] Проверено на семинаре


In [53]:
sparql = SPARQLWrapper("https://dbpedia.org/sparql")
sparql.setReturnFormat(JSON)

In [54]:
query = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?dbpediaEntity ?cityLabel ?populationDBpedia ?wikidataLink
WHERE {
  ?dbpediaEntity a dbo:City .
  ?dbpediaEntity dbo:country dbr:Russia .
  OPTIONAL { ?dbpediaEntity dbo:populationTotal ?populationDBpedia }
  ?dbpediaEntity owl:sameAs ?wikidataLink .
  ?dbpediaEntity rdfs:label ?cityLabel .

  FILTER(lang(?cityLabel)="en")
  FILTER(STRSTARTS(STR(?wikidataLink), "http://www.wikidata.org/entity/"))
}
ORDER BY DESC(?populationDBpedia)
LIMIT 20
"""

In [56]:
sparql.setQuery(query)
results = sparql.query().convert()

data = []
for r in results["results"]["bindings"]:
    city = r["cityLabel"]["value"]
    dbpedia = r["dbpediaEntity"]["value"]
    pop_dbp = r.get("populationDBpedia", {}).get("value", "")
    wikidata = r["wikidataLink"]["value"]
    data.append([city, dbpedia, pop_dbp, wikidata])

df_dbpedia = pd.DataFrame(
    data, columns=["City", "DBpedia", "Population_DBpedia", "Wikidata"]
)
df_dbpedia.head()


,City,DBpedia,Population_DBpedia,Wikidata
0,Saint Petersburg,http://dbpedia.org/resource/Saint_Petersburg,5601911,http://www.wikidata.org/entity/Q656
1,Saint Petersburg,http://dbpedia.org/resource/Saint_Petersburg,5601911,http://www.wikidata.org/entity/Q909810
2,Luhansk,http://dbpedia.org/resource/Luhansk,397677,http://www.wikidata.org/entity/Q134279
3,Simferopol,http://dbpedia.org/resource/Simferopol,332317,http://www.wikidata.org/entity/Q19566
4,Simferopol,http://dbpedia.org/resource/Simferopol,332317,http://www.wikidata.org/entity/Q2237983


In [57]:
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)
wikidata_entities = df_dbpedia["Wikidata"].tolist()
values_clause = " ".join(f"wd:{url.split('/')[-1]}" for url in wikidata_entities)

query = f"""
SELECT ?city ?cityLabel ?populationWikidata
WHERE {{
  VALUES ?city {{ {values_clause} }}
  ?city wdt:P1082 ?populationWikidata .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
"""


In [58]:
values_clause

'wd:Q656 wd:Q909810 wd:Q134279 wd:Q19566 wd:Q2237983 wd:Q6948367 wd:Q157260 wd:Q157065 wd:Q706857 wd:Q165413 wd:Q161978 wd:Q17266378 wd:Q158604 wd:Q21485205 wd:Q161987 wd:Q863330 wd:Q997522 wd:Q5168 wd:Q3752 wd:Q4303410'

In [59]:
sparql.setQuery(query)
results = sparql.query().convert()

data = []
for r in results["results"]["bindings"]:
    city = r["cityLabel"]["value"]
    pop_wd = r["populationWikidata"]["value"]
    data.append([city, pop_wd])

df_wikidata = pd.DataFrame(data, columns=["City", "Population_Wikidata"])

# Объединяем с DBpedia
df_final = pd.merge(df_dbpedia, df_wikidata, on="City", how="left"); df_final.head()

,City,DBpedia,Population_DBpedia,Wikidata,Population_Wikidata
0,Saint Petersburg,http://dbpedia.org/resource/Saint_Petersburg,5601911,http://www.wikidata.org/entity/Q656,5652922
1,Saint Petersburg,http://dbpedia.org/resource/Saint_Petersburg,5601911,http://www.wikidata.org/entity/Q909810,5652922
2,Luhansk,http://dbpedia.org/resource/Luhansk,397677,http://www.wikidata.org/entity/Q134279,417990
3,Simferopol,http://dbpedia.org/resource/Simferopol,332317,http://www.wikidata.org/entity/Q19566,335009
4,Simferopol,http://dbpedia.org/resource/Simferopol,332317,http://www.wikidata.org/entity/Q2237983,335009
